In [1]:
!pip install -q torch torchvision gradio scikit-learn seaborn matplotlib Pillow
!pip install split-folders

1. Import Libraries

In [2]:
import os
import json
import copy
import time
import random
import shutil
import warnings
import argparse
import splitfolders
import os
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

warnings.filterwarnings('ignore')
print('✅  Imports done')

✅  Imports done


2.   Config & Constants

In [3]:
# ── Kaggle dataset path (attached via the dataset link above) ──────────────────
# Dataset: https://www.kaggle.com/datasets/abdullahsaood/pakistani-politicians-images-dataset
DATASET_PATH = Path('/kaggle/input/datasets/abdullahsaood/pakistani-politicians-images-dataset/Politician Dataset')
OUTPUT_PATH  = Path('/kaggle/working/dataset_split')
RESULTS_DIR  = Path('/kaggle/working/results')

# ── Class map: snake_case label → actual folder name on disk ──────────────────
CLASSES = {
    'ahsan_iqbal':         'Ahsan Iqbal',
    'asif_zardari':        'Asif Zardari',
    'benazir_bhutto':      'Benazir Bhutto',
    'bilawal_bhutto':      'Bilawal Bhutto',
    'hamza_shehbaz':       'Hamza Shehbaz',
    'imran_khan':          'Imran Khan',
    'ishaq_dar':           'Ishaq Dar',
    'khawaja_asif':        'Khawaja_Asif',
    'maryam_nawaz':        'Maryam Nawaz',
    'mohsin_naqvi':        'Mohsin naqvi',
    'murad_ali_shah':      'Murad Ali Shah',
    'nawaz_sharif':        'Nawaz Sharif',
    'pervez_musharraf':    'Pervez_Musharraf',
    'rana_sanaullah':      'Rana Sanaullah',
    'shehbaz_sharif':      'Shehbaz Sharif',
    'yousef_raza_gillani': 'Yousef Raza Gillani',
}

CLASS_LABELS = list(CLASSES.keys())   # ordered list of snake_case labels
NUM_CLASSES  = len(CLASSES)           # 16

# ── Split ratios ───────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.75
VAL_RATIO   = 0.15
TEST_RATIO  = 0.10
SEED        = 42

# ── Training hyper-params ──────────────────────────────────────────────────────
IMG_SIZE       = 224
BATCH_SIZE     = 32
NUM_EPOCHS     = 30
LR             = 1e-4
WEIGHT_DECAY   = 1e-4
NUM_WORKERS    = 2
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]

try:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if DEVICE.type == "cuda":
        x = torch.randn(1).to(DEVICE)
        print("CUDA is working")

except Exception as e:
    print("CUDA issue detected:", e)
    DEVICE = torch.device("cpu")

print("Using device:", DEVICE)
random.seed(SEED)

print(f'Device  : {DEVICE}')
print(f'Classes : {NUM_CLASSES}')
print(f'Dataset : {DATASET_PATH}')

CUDA is working
Using device: cuda
Device  : cuda
Classes : 16
Dataset : /kaggle/input/datasets/abdullahsaood/pakistani-politicians-images-dataset/Politician Dataset


3.  Dataset Split

In [4]:
def split_dataset(src_root: Path, dst_root: Path) -> None:
    """
    Read class folders from src_root (using actual folder names),
    split files, and write to dst_root/{train,val,test}/<snake_case_label>/.
    """
    src_root = Path(src_root)
    dst_root = Path(dst_root)

    for split in ('train', 'val', 'test'):
        (dst_root / split).mkdir(parents=True, exist_ok=True)

    for label, folder_name in CLASSES.items():
        cls_path = src_root / folder_name

        if not cls_path.exists():
            print(f'[WARN] Missing class folder: {cls_path} — skipping')
            continue

        images = [
            f for f in cls_path.iterdir()
            if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}
        ]
        random.shuffle(images)

        n       = len(images)
        n_train = int(n * TRAIN_RATIO)
        n_val   = int(n * VAL_RATIO)

        splits = {
            'train': images[:n_train],
            'val':   images[n_train : n_train + n_val],
            'test':  images[n_train + n_val :],
        }

        for split_name, files in splits.items():
            out_dir = dst_root / split_name / label   # snake_case output folder
            out_dir.mkdir(parents=True, exist_ok=True)
            for f in files:
                shutil.copy2(f, out_dir / f.name)

        print(f'[{label}]  total={n}  train={len(splits["train"])}  '
              f'val={len(splits["val"])}  test={len(splits["test"])}')

    print(f'\n✅  Dataset split saved to: {dst_root}')


split_dataset(src_root=DATASET_PATH, dst_root=OUTPUT_PATH)

[ahsan_iqbal]  total=84  train=63  val=12  test=9
[asif_zardari]  total=84  train=63  val=12  test=9
[benazir_bhutto]  total=142  train=106  val=21  test=15
[bilawal_bhutto]  total=12  train=9  val=1  test=2
[hamza_shehbaz]  total=80  train=60  val=12  test=8
[imran_khan]  total=102  train=76  val=15  test=11
[ishaq_dar]  total=82  train=61  val=12  test=9
[khawaja_asif]  total=94  train=70  val=14  test=10
[maryam_nawaz]  total=80  train=60  val=12  test=8
[mohsin_naqvi]  total=28  train=21  val=4  test=3
[murad_ali_shah]  total=84  train=63  val=12  test=9
[nawaz_sharif]  total=93  train=69  val=13  test=11
[pervez_musharraf]  total=95  train=71  val=14  test=10
[rana_sanaullah]  total=88  train=66  val=13  test=9
[shehbaz_sharif]  total=88  train=66  val=13  test=9
[yousef_raza_gillani]  total=85  train=63  val=12  test=10

✅  Dataset split saved to: /kaggle/working/dataset_split


4.  Class Distribution Check

In [5]:
def print_class_distribution(dataset_root: Path) -> None:
    """Print per-class image counts for each split."""
    root = Path(dataset_root)
    print('\n── Dataset Distribution ──────────────────────────')
    print(f'{"Class":<35} {"Train":>6} {"Val":>5} {"Test":>5} {"Total":>7}')
    print('─' * 58)
    for cls in sorted(os.listdir(root / 'train')):
        tr = len(list((root / 'train' / cls).glob('*')))
        vl = len(list((root / 'val'   / cls).glob('*'))) if (root / 'val' / cls).exists() else 0
        te = len(list((root / 'test'  / cls).glob('*'))) if (root / 'test'/ cls).exists() else 0
        print(f'{cls:<35} {tr:>6} {vl:>5} {te:>5} {tr+vl+te:>7}')
    print('─' * 58)


print_class_distribution(OUTPUT_PATH)



── Dataset Distribution ──────────────────────────
Class                                Train   Val  Test   Total
──────────────────────────────────────────────────────────
ahsan_iqbal                             63    12     9      84
asif_zardari                            63    12     9      84
benazir_bhutto                         106    21    15     142
bilawal_bhutto                           9     1     2      12
hamza_shehbaz                           60    12     8      80
imran_khan                              76    15    11     102
ishaq_dar                               61    12     9      82
khawaja_asif                            70    14    10      94
maryam_nawaz                            60    12     8      80
mohsin_naqvi                            21     4     3      28
murad_ali_shah                          63    12     9      84
nawaz_sharif                            69    13    11      93
pervez_musharraf                        71    14    10      95
rana_sa

In [6]:
from pathlib import Path

input_root = Path('/kaggle/input')

print("Contents of /kaggle/input:\n")

for p in input_root.iterdir():
    print(p)

    if p.is_dir():
        print(" Subfolders:")
        for sub in p.iterdir():
            print("   ", sub.name)

Contents of /kaggle/input:

/kaggle/input/datasets
 Subfolders:
    abdullahsaood


5.  Transforms & DataLoaders

In [7]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def get_dataloaders(dataset_root: Path = OUTPUT_PATH,
                    batch_size: int = BATCH_SIZE,
                    num_workers: int = NUM_WORKERS):
    """Returns (train_loader, val_loader, test_loader, class_names)."""
    root = Path(dataset_root)

    train_ds = datasets.ImageFolder(root / 'train', transform=train_transform)
    val_ds   = datasets.ImageFolder(root / 'val',   transform=val_test_transform)
    test_ds  = datasets.ImageFolder(root / 'test',  transform=val_test_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=num_workers, pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=num_workers, pin_memory=False)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, num_workers=num_workers, pin_memory=False)

    print(f'Classes ({len(train_ds.classes)}): {train_ds.classes}')
    print(f'Train: {len(train_ds)}  |  Val: {len(val_ds)}  |  Test: {len(test_ds)}')
    return train_loader, val_loader, test_loader, train_ds.classes


train_loader, val_loader, test_loader, class_names = get_dataloaders()

images, labels = next(iter(train_loader))
print(f'\nSample batch — images: {images.shape}  labels: {labels.shape}')

Classes (16): ['ahsan_iqbal', 'asif_zardari', 'benazir_bhutto', 'bilawal_bhutto', 'hamza_shehbaz', 'imran_khan', 'ishaq_dar', 'khawaja_asif', 'maryam_nawaz', 'mohsin_naqvi', 'murad_ali_shah', 'nawaz_sharif', 'pervez_musharraf', 'rana_sanaullah', 'shehbaz_sharif', 'yousef_raza_gillani']
Train: 987  |  Val: 192  |  Test: 142

Sample batch — images: torch.Size([32, 3, 224, 224])  labels: torch.Size([32])


6.  Model Builders 

In [8]:
def build_resnet50(num_classes: int = NUM_CLASSES, freeze_backbone: bool = False) -> nn.Module:
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.fc.in_features   # 2048
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes),
    )
    return model


def build_efficientnet_b2(num_classes: int = NUM_CLASSES, freeze_backbone: bool = False) -> nn.Module:
    model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.classifier[1].in_features   # 1408
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes),
    )
    return model


MODEL_REGISTRY = {
    'resnet50':        build_resnet50,
    'efficientnet_b2': build_efficientnet_b2,
}

print('✅  Model builders registered:', list(MODEL_REGISTRY.keys()))

✅  Model builders registered: ['resnet50', 'efficientnet_b2']


7. Training Loop 

In [9]:
def plot_curves(history: dict, save_dir: Path, model_name: str) -> None:
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Training Curves — {model_name}', fontsize=14, fontweight='bold')

    ax1.plot(epochs, history['train_loss'], 'b-o', markersize=3, label='Train Loss')
    ax1.plot(epochs, history['val_loss'],   'r-o', markersize=3, label='Val Loss')
    ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(epochs, history['train_acc'], 'b-o', markersize=3, label='Train Acc')
    ax2.plot(epochs, history['val_acc'],   'r-o', markersize=3, label='Val Acc')
    ax2.axhline(y=90, color='g', linestyle='--', linewidth=1.2, label='90% target')
    ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_dir / 'training_curves.png', dpi=150)
    plt.show()
    print(f'📊  Curves saved → {save_dir / "training_curves.png"}')


@torch.no_grad()
def evaluate(model: nn.Module, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss    = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total * 100


def train_model(model: nn.Module, train_loader, val_loader,
                num_epochs: int = NUM_EPOCHS, lr: float = LR,
                weight_decay: float = WEIGHT_DECAY,
                save_dir: Path = RESULTS_DIR, model_name: str = 'model'):
    model      = model.to(DEVICE)
    save_dir   = Path(save_dir) / model_name
    save_dir.mkdir(parents=True, exist_ok=True)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=weight_decay
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    history      = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    print(f'\n{"="*60}')
    print(f'  Training: {model_name.upper()}  |  Device: {DEVICE}')
    print(f'  Epochs: {num_epochs}  |  LR: {lr}  |  Classes: {NUM_CLASSES}')
    print(f'{"="*60}\n')

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        model.train()
        running_loss, running_correct, total = 0.0, 0, 0

        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss    = criterion(outputs, lbls)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss    += loss.item() * imgs.size(0)
            preds            = outputs.argmax(dim=1)
            running_correct += (preds == lbls).sum().item()
            total           += imgs.size(0)

        train_loss = running_loss / total
        train_acc  = running_correct / total * 100
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, save_dir / 'best_model.pt')

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f'Epoch [{epoch:02d}/{num_epochs}]  '
              f'Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  |  '
              f'Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%  '
              f'(best: {best_val_acc:.2f}%)  [{time.time()-t0:.1f}s]')

    model.load_state_dict(best_weights)
    print(f'\n✅  Training complete. Best Val Acc: {best_val_acc:.2f}%')

    with open(save_dir / 'history.json', 'w') as f:
        json.dump(history, f, indent=2)

    plot_curves(history, save_dir, model_name)
    return model, history


print('✅  Training functions defined')

✅  Training functions defined


In [10]:
from PIL import Image, ImageFile
from pathlib import Path

# Allow truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

def remove_bad_images(dataset_path):
    dataset_path = Path(dataset_path)

    removed = 0

    for img_path in dataset_path.rglob("*"):

        if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:

            try:
                with Image.open(img_path) as img:
                    img.verify()

            except Exception as e:
                print(f"Removing corrupted image: {img_path}")
                img_path.unlink()

                removed += 1

    print(f"\nTotal corrupted images removed: {removed}")

# Run cleaning
remove_bad_images('/kaggle/working/dataset_split')

# Recreate datasets and dataloaders after cleaning images

train_loader, val_loader, test_loader, class_names = get_dataloaders()

print("DataLoaders rebuilt successfully")

Removing corrupted image: /kaggle/working/dataset_split/train/yousef_raza_gillani/File-Yousaf_Raza_Gilani_2010_-cropped-.jpg

Total corrupted images removed: 1
Classes (16): ['ahsan_iqbal', 'asif_zardari', 'benazir_bhutto', 'bilawal_bhutto', 'hamza_shehbaz', 'imran_khan', 'ishaq_dar', 'khawaja_asif', 'maryam_nawaz', 'mohsin_naqvi', 'murad_ali_shah', 'nawaz_sharif', 'pervez_musharraf', 'rana_sanaullah', 'shehbaz_sharif', 'yousef_raza_gillani']
Train: 986  |  Val: 192  |  Test: 142
DataLoaders rebuilt successfully


8. ResNet 50

In [11]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

resnet_model = build_resnet50(freeze_backbone=False)

# Move model to GPU
resnet_model = resnet_model.to(DEVICE)

print("Using device:", DEVICE)

resnet_model, resnet_history = train_model(
    model        = resnet_model,
    train_loader = train_loader,
    val_loader   = val_loader,
    num_epochs   = NUM_EPOCHS,
    lr           = LR,
    save_dir     = RESULTS_DIR,
    model_name   = 'resnet50',
)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 195MB/s]


Using device: cuda

  Training: RESNET50  |  Device: cuda
  Epochs: 30  |  LR: 0.0001  |  Classes: 16

Epoch [01/30]  Train Loss: 2.6932  Train Acc: 13.29%  |  Val Loss: 2.4868  Val Acc: 25.52%  (best: 25.52%)  [12.5s]
Epoch [02/30]  Train Loss: 2.3457  Train Acc: 35.80%  |  Val Loss: 1.9425  Val Acc: 59.90%  (best: 59.90%)  [11.0s]
Epoch [03/30]  Train Loss: 1.7802  Train Acc: 60.55%  |  Val Loss: 1.3351  Val Acc: 75.52%  (best: 75.52%)  [11.5s]
Epoch [04/30]  Train Loss: 1.2386  Train Acc: 79.41%  |  Val Loss: 0.9942  Val Acc: 84.90%  (best: 84.90%)  [11.5s]
Epoch [05/30]  Train Loss: 0.9469  Train Acc: 87.53%  |  Val Loss: 0.8943  Val Acc: 88.54%  (best: 88.54%)  [11.4s]
Epoch [06/30]  Train Loss: 0.8132  Train Acc: 93.41%  |  Val Loss: 0.8103  Val Acc: 93.23%  (best: 93.23%)  [12.0s]
Epoch [07/30]  Train Loss: 0.7415  Train Acc: 95.54%  |  Val Loss: 0.8229  Val Acc: 91.15%  (best: 93.23%)  [11.6s]
Epoch [08/30]  Train Loss: 0.7028  Train Acc: 96.86%  |  Val Loss: 0.7843  Val Acc: 9

9.EffcientNet-B2

In [14]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

effnet_model = build_efficientnet_b2(freeze_backbone=False)

# Move model to GPU
effnet_model = effnet_model.to(DEVICE)

print("Using device:", DEVICE)

effnet_model, effnet_history = train_model(
    model        = effnet_model,
    train_loader = train_loader,
    val_loader   = val_loader,
    num_epochs   = NUM_EPOCHS,
    lr           = LR,
    save_dir     = RESULTS_DIR,
    model_name   = 'efficientnet_b2',
)

Using device: cuda

  Training: EFFICIENTNET_B2  |  Device: cuda
  Epochs: 30  |  LR: 0.0001  |  Classes: 16

Epoch [01/30]  Train Loss: 2.7195  Train Acc: 16.23%  |  Val Loss: 2.6244  Val Acc: 36.98%  (best: 36.98%)  [11.1s]
Epoch [02/30]  Train Loss: 2.4908  Train Acc: 36.82%  |  Val Loss: 2.2433  Val Acc: 53.65%  (best: 53.65%)  [10.6s]
Epoch [03/30]  Train Loss: 2.1536  Train Acc: 53.04%  |  Val Loss: 1.8313  Val Acc: 66.15%  (best: 66.15%)  [11.0s]
Epoch [04/30]  Train Loss: 1.7486  Train Acc: 66.02%  |  Val Loss: 1.4316  Val Acc: 75.52%  (best: 75.52%)  [11.0s]
Epoch [05/30]  Train Loss: 1.3514  Train Acc: 78.50%  |  Val Loss: 1.1366  Val Acc: 85.42%  (best: 85.42%)  [10.7s]
Epoch [06/30]  Train Loss: 1.0880  Train Acc: 86.11%  |  Val Loss: 1.0090  Val Acc: 86.46%  (best: 86.46%)  [10.4s]
Epoch [07/30]  Train Loss: 0.9291  Train Acc: 90.26%  |  Val Loss: 0.8934  Val Acc: 90.62%  (best: 90.62%)  [10.7s]
Epoch [08/30]  Train Loss: 0.8384  Train Acc: 92.19%  |  Val Loss: 0.8599  Val

10. Evaluation Functions

In [15]:
@torch.no_grad()
def get_predictions(model: nn.Module, loader):
    """Returns (all_labels, all_preds, all_probs, all_paths)."""
    model.eval()
    all_labels, all_preds, all_probs, all_paths = [], [], [], []

    for batch_idx, (images, labels) in enumerate(loader):
        images  = images.to(DEVICE)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1).cpu()
        preds   = probs.argmax(dim=1)

        all_labels.extend(labels.numpy())
        all_preds.extend(preds.numpy())
        all_probs.extend(probs.numpy())

        start = batch_idx * loader.batch_size
        end   = start + images.size(0)
        for idx in range(start, min(end, len(loader.dataset))):
            all_paths.append(loader.dataset.samples[idx][0])

    return np.array(all_labels), np.array(all_preds), np.array(all_probs), all_paths


def plot_confusion_matrix(y_true, y_pred, class_names, save_dir: Path, normalize: bool = True):
    cm = confusion_matrix(y_true, y_pred)
    cm_plot = cm.astype(float) / cm.sum(axis=1, keepdims=True) if normalize else cm
    fmt     = '.2f' if normalize else 'd'
    title   = 'Confusion Matrix (Normalized)' if normalize else 'Confusion Matrix (Counts)'

    short_names = [n.replace('_', '\n') for n in class_names]
    fig, ax = plt.subplots(figsize=(18, 15))
    sns.heatmap(cm_plot, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=short_names, yticklabels=short_names,
                linewidths=0.4, linecolor='lightgrey', ax=ax,
                cbar_kws={'shrink': 0.8})
    ax.set_title(title, fontsize=16, fontweight='bold', pad=15)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.tick_params(axis='x', labelsize=7, rotation=45)
    ax.tick_params(axis='y', labelsize=7, rotation=0)
    plt.tight_layout()
    out_path = save_dir / 'confusion_matrix.png'
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f'📊  Confusion matrix saved → {out_path}')


def plot_top5_misclassified(y_true, y_pred, all_probs, all_paths, class_names, save_dir: Path):
    wrong_mask = y_true != y_pred
    wrong_idx  = np.where(wrong_mask)[0]
    if len(wrong_idx) == 0:
        print('🎉  No misclassifications found!')
        return
    wrong_conf = np.array([all_probs[i][y_pred[i]] for i in wrong_idx])
    top5_idx   = wrong_idx[np.argsort(-wrong_conf)[:5]]

    fig, axes = plt.subplots(1, len(top5_idx), figsize=(4 * len(top5_idx), 5))
    if len(top5_idx) == 1:
        axes = [axes]
    fig.suptitle('Top-5 Most Confident Misclassifications', fontsize=14, fontweight='bold')

    for ax, idx in zip(axes, top5_idx):
        img       = Image.open(all_paths[idx]).convert('RGB')
        true_name = class_names[y_true[idx]].replace('_', ' ').title()
        pred_name = class_names[y_pred[idx]].replace('_', ' ').title()
        conf      = all_probs[idx][y_pred[idx]] * 100
        ax.imshow(img)
        ax.set_title(f'True: {true_name}\nPred: {pred_name}\nConf: {conf:.1f}%',
                     fontsize=8, color='red')
        ax.axis('off')

    plt.tight_layout()
    out_path = save_dir / 'top5_misclassified.png'
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f'🔍  Top-5 misclassified saved → {out_path}')


def full_evaluation(model: nn.Module, test_loader, class_names: list,
                    save_dir: Path = RESULTS_DIR / 'model') -> dict:
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'  Evaluating on test set  |  Device: {DEVICE}')
    print(f'{"="*60}\n')

    y_true, y_pred, all_probs, all_paths = get_predictions(model, test_loader)

    overall_acc = accuracy_score(y_true, y_pred) * 100
    prec, rec, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=list(range(len(class_names)))
    )
    macro_prec, macro_rec, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro'
    )

    print(f'Overall Accuracy : {overall_acc:.2f}%')
    print(f'Macro Precision  : {macro_prec*100:.2f}%')
    print(f'Macro Recall     : {macro_rec*100:.2f}%')
    print(f'Macro F1         : {macro_f1*100:.2f}%\n')
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    if overall_acc < 90:
        print('⚠️   Accuracy below 90% target — consider more epochs or data.')
    else:
        print('✅   90%+ accuracy target achieved!')

    per_class = {cls: {'precision': float(prec[i]), 'recall': float(rec[i]),
                       'f1': float(f1[i]), 'support': int(support[i])}
                 for i, cls in enumerate(class_names)}

    summary = {
        'overall_accuracy': round(overall_acc, 4),
        'macro_precision':  round(float(macro_prec) * 100, 4),
        'macro_recall':     round(float(macro_rec)  * 100, 4),
        'macro_f1':         round(float(macro_f1)   * 100, 4),
        'per_class_metrics': per_class,
    }

    with open(save_dir / 'evaluation_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    print(f'\n📄  Evaluation JSON saved → {save_dir / "evaluation_summary.json"}')

    plot_confusion_matrix(y_true, y_pred, class_names, save_dir, normalize=True)
    plot_top5_misclassified(y_true, y_pred, all_probs, all_paths, class_names, save_dir)

    return summary


print('✅  Evaluation functions defined')

✅  Evaluation functions defined


11. Evaluate ResNet 50

In [16]:
resnet_summary = full_evaluation(
    model        = resnet_model,
    test_loader  = test_loader,
    class_names  = class_names,
    save_dir     = RESULTS_DIR / 'resnet50',
)


  Evaluating on test set  |  Device: cuda

Overall Accuracy : 88.73%
Macro Precision  : 90.87%
Macro Recall     : 85.91%
Macro F1         : 87.12%

                     precision    recall  f1-score   support

        ahsan_iqbal     0.9000    1.0000    0.9474         9
       asif_zardari     1.0000    0.7778    0.8750         9
     benazir_bhutto     0.8667    0.8667    0.8667        15
     bilawal_bhutto     1.0000    0.5000    0.6667         2
      hamza_shehbaz     1.0000    0.8750    0.9333         8
         imran_khan     0.9091    0.9091    0.9091        11
          ishaq_dar     0.7778    0.7778    0.7778         9
       khawaja_asif     1.0000    1.0000    1.0000        10
       maryam_nawaz     0.7000    0.8750    0.7778         8
       mohsin_naqvi     1.0000    0.6667    0.8000         3
     murad_ali_shah     0.7273    0.8889    0.8000         9
       nawaz_sharif     1.0000    0.9091    0.9524        11
   pervez_musharraf     0.9091    1.0000    0.9524       

12. Evaluate EfficientNet-B2

In [17]:
effnet_summary = full_evaluation(
    model        = effnet_model,
    test_loader  = test_loader,
    class_names  = class_names,
    save_dir     = RESULTS_DIR / 'efficientnet_b2',
)


  Evaluating on test set  |  Device: cuda

Overall Accuracy : 87.32%
Macro Precision  : 83.62%
Macro Recall     : 81.84%
Macro F1         : 81.91%

                     precision    recall  f1-score   support

        ahsan_iqbal     0.8000    0.8889    0.8421         9
       asif_zardari     1.0000    1.0000    1.0000         9
     benazir_bhutto     0.8750    0.9333    0.9032        15
     bilawal_bhutto     0.0000    0.0000    0.0000         2
      hamza_shehbaz     1.0000    0.6250    0.7692         8
         imran_khan     0.7692    0.9091    0.8333        11
          ishaq_dar     0.8889    0.8889    0.8889         9
       khawaja_asif     1.0000    1.0000    1.0000        10
       maryam_nawaz     0.7778    0.8750    0.8235         8
       mohsin_naqvi     1.0000    0.6667    0.8000         3
     murad_ali_shah     0.7500    1.0000    0.8571         9
       nawaz_sharif     1.0000    0.8182    0.9000        11
   pervez_musharraf     1.0000    0.9000    0.9474       

13. Model Comparison

In [18]:
def compare_models(results_dir: Path = RESULTS_DIR) -> None:
    results_dir = Path(results_dir)
    summaries   = {}

    for model_dir in sorted(results_dir.iterdir()):
        summary_path = model_dir / 'evaluation_summary.json'
        if summary_path.exists():
            with open(summary_path) as f:
                summaries[model_dir.name] = json.load(f)

    if not summaries:
        print('No evaluation summaries found.')
        return

    print('\n── Model Comparison ──────────────────────────────────────')
    print(f'{"Model":<25} {"Accuracy":>10} {"Macro P":>10} {"Macro R":>10} {"Macro F1":>10}')
    print('─' * 67)
    for name, s in summaries.items():
        print(f'{name:<25} {s["overall_accuracy"]:>9.2f}% '
              f'{s["macro_precision"]:>9.2f}% '
              f'{s["macro_recall"]:>9.2f}% '
              f'{s["macro_f1"]:>9.2f}%')
    print('─' * 67)

    metrics = ['overall_accuracy', 'macro_precision', 'macro_recall', 'macro_f1']
    labels  = ['Accuracy', 'Precision', 'Recall', 'F1']
    x       = np.arange(len(labels))
    width   = 0.35 / max(len(summaries), 1)

    fig, ax = plt.subplots(figsize=(10, 6))
    for i, (name, s) in enumerate(summaries.items()):
        ax.bar(x + i * width, [s[m] for m in metrics], width, label=name)

    ax.set_title('Model Comparison — Test Set Metrics', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width * (len(summaries) - 1) / 2)
    ax.set_xticklabels(labels)
    ax.set_ylabel('Score (%)')
    ax.set_ylim(0, 105)
    ax.axhline(y=90, color='green', linestyle='--', label='90% target')
    ax.legend(); ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    out = results_dir / 'model_comparison.png'
    plt.savefig(out, dpi=150)
    plt.show()
    print(f'\n📊  Comparison chart saved → {out}')


compare_models(RESULTS_DIR)


── Model Comparison ──────────────────────────────────────
Model                       Accuracy    Macro P    Macro R   Macro F1
───────────────────────────────────────────────────────────────────
efficientnet_b2               87.32%     83.62%     81.84%     81.91%
resnet50                      88.73%     90.87%     85.91%     87.12%
───────────────────────────────────────────────────────────────────

📊  Comparison chart saved → /kaggle/working/results/model_comparison.png


14. Gradio Inference App

In [19]:
import gradio as gr

DISPLAY_NAMES = {
    'ahsan_iqbal':         'Ahsan Iqbal',
    'asif_zardari':        'Asif Ali Zardari',
    'benazir_bhutto':      'Benazir Bhutto',
    'bilawal_bhutto':      'Bilawal Bhutto Zardari',
    'hamza_shehbaz':       'Hamza Shehbaz',
    'imran_khan':          'Imran Khan',
    'ishaq_dar':           'Ishaq Dar',
    'khawaja_asif':        'Khawaja Asif',
    'maryam_nawaz':        'Maryam Nawaz',
    'mohsin_naqvi':        'Mohsin Naqvi',
    'murad_ali_shah':      'Murad Ali Shah',
    'nawaz_sharif':        'Nawaz Sharif',
    'pervez_musharraf':    'Pervez Musharraf',
    'rana_sanaullah':      'Rana Sanaullah',
    'shehbaz_sharif':      'Shehbaz Sharif',
    'yousef_raza_gillani': 'Yousef Raza Gillani',
}

AVAILABLE_MODELS = {
    'ResNet-50':       ('resnet50',        RESULTS_DIR / 'resnet50'       / 'best_model.pt'),
    'EfficientNet-B2': ('efficientnet_b2', RESULTS_DIR / 'efficientnet_b2'/ 'best_model.pt'),
}

_model_cache: dict = {}

infer_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def load_model_cached(display_name: str):
    if display_name in _model_cache:
        return _model_cache[display_name]
    model_key, ckpt_path = AVAILABLE_MODELS[display_name]
    if not Path(ckpt_path).exists():
        return None
    model = MODEL_REGISTRY[model_key](num_classes=NUM_CLASSES)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval().to(DEVICE)
    _model_cache[display_name] = model
    return model


@torch.no_grad()
def predict(image, model_choice: str):
    model = load_model_cached(model_choice)
    if model is None:
        return {}, '⚠️ Checkpoint not found. Train the model first.', None
    if image is None:
        return {}, 'Please upload an image.', None

    img_tensor = infer_transform(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    probs      = torch.softmax(model(img_tensor), dim=1).squeeze().cpu().numpy()
    top5_idx   = np.argsort(-probs)[:5]
    top1_idx   = top5_idx[0]
    top1_name  = DISPLAY_NAMES.get(class_names[top1_idx], class_names[top1_idx])
    top1_conf  = probs[top1_idx] * 100

    conf_dict = {
        DISPLAY_NAMES.get(class_names[i], class_names[i]): float(probs[i])
        for i in top5_idx
    }

    info = (f'### 🏆 Predicted: **{top1_name}**\n'
            f'**Confidence:** {top1_conf:.2f}%\n\n'
            f'**Model used:** {model_choice}\n\n'
            f'**Top-5 predictions:**\n')
    for rank, i in enumerate(top5_idx, 1):
        info += f'{rank}. {DISPLAY_NAMES.get(class_names[i], class_names[i])} — {probs[i]*100:.2f}%\n'

    fig, ax = plt.subplots(figsize=(7, 4))
    names  = [DISPLAY_NAMES.get(class_names[i], class_names[i]).replace(' ', '\n') for i in top5_idx]
    colors = ['#2563EB'] + ['#93C5FD'] * 4
    bars   = ax.barh(names[::-1], [probs[i]*100 for i in top5_idx[::-1]],
                     color=colors[::-1], edgecolor='white')
    ax.set_xlabel('Confidence (%)')
    ax.set_title('Top-5 Predictions', fontweight='bold')
    ax.set_xlim(0, 100)
    for bar, val in zip(bars, [probs[i]*100 for i in top5_idx[::-1]]):
        ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()

    return conf_dict, info, fig


CSS = """
body { font-family: 'Segoe UI', sans-serif; }
.title { text-align: center; color: #1e3a5f; }
.subtitle { text-align: center; color: #666; font-size: 0.95rem; }
footer { display: none !important; }
"""

with gr.Blocks(css=CSS, title='Pakistani Politician Classifier',
               theme=gr.themes.Soft(primary_hue='blue')) as demo:

    gr.Markdown("""
        <h1 class='title'>🇵🇰 Pakistani Politician Image Classifier</h1>
        <p class='subtitle'>
            Deep Learning — Project 2 (Category B) &nbsp;|&nbsp;
            ResNet-50 &amp; EfficientNet-B2 &nbsp;|&nbsp; 16 Classes
        </p>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            image_input  = gr.Image(type='pil', label='Upload Politician Image')
            model_choice = gr.Radio(choices=list(AVAILABLE_MODELS.keys()),
                                    value='ResNet-50', label='Select Model')
            predict_btn  = gr.Button('🔍  Classify', variant='primary', size='lg')

        with gr.Column(scale=2):
            info_output  = gr.Markdown(label='Prediction')
            label_output = gr.Label(num_top_classes=5, label='Top-5 Confidence')
            chart_output = gr.Plot(label='Confidence Chart')

    predict_btn.click(
        fn=predict,
        inputs=[image_input, model_choice],
        outputs=[label_output, info_output, chart_output],
    )

    gr.Markdown("""
        ---
        **Classes (16):** Ahsan Iqbal · Asif Zardari · Benazir Bhutto · Bilawal Bhutto ·
        Hamza Shehbaz · Imran Khan · Ishaq Dar · Khawaja Asif · Maryam Nawaz ·
        Mohsin Naqvi · Murad Ali Shah · Nawaz Sharif · Pervez Musharraf ·
        Rana Sanaullah · Shehbaz Sharif · Yousef Raza Gillani
    """)

demo.launch(share=True)   # share=True gives a public URL on Kaggle

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://44acc10fe3982d5fa7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


###### 